# Part 13: Transaction Cost Sensitivity

This notebook runs the reproducible Part 13 Python runner in Colab. Part 13 implements the required Part 10 benchmark scenarios under low/medium/high transaction costs and reports both excluding and including initial formation cost.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

assert (PROJECT_DIR / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_DIR / 'experiments' / 'part13_transaction_cost_sensitivity' / 'run_part13_transaction_cost_sensitivity.py').exists(), 'Part 13 runner not found.'

In [ ]:
!python -m pip install -q -r experiments/part13_transaction_cost_sensitivity/requirements-part13.txt

## Path configuration

Part 13 requires completed Part 10 and Part 5 runs. The helper first tries the canonical Drive path, then the downloaded `*_outputs/...` folder structure.

In [ ]:
from pathlib import Path

def choose_run_dir(run_id, required_files, *candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists() and all((path / item).exists() for item in required_files):
            return path
    roots = [Path('outputs'), Path('.')]
    matches = []
    for root in roots:
        if root.exists():
            for path in root.rglob(run_id):
                if path.is_dir() and all((path / item).exists() for item in required_files):
                    matches.append(path)
    if matches:
        matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
        print(f'Auto-detected {run_id}:', matches[0])
        return matches[0]
    print(f'Could not find {run_id}. Candidate directories containing this name:')
    for root in roots:
        if root.exists():
            for path in root.rglob(run_id):
                print('  ', path)
    return Path(candidates[0])

INPUT_DIR = Path('data_2026/cleaned')
PART10_RUN_DIR = choose_run_dir(
    'colab_part10_seed42',
    ['run_manifest.json', 'results/output_validation_summary.json', 'results/part10_weekly_weights.csv', 'results/part10_scenario_dictionary.csv'],
    'outputs/part10_benchmark_cap_sensitivity/colab_part10_seed42',
    'outputs/part10_benchmark_cap_sensitivity/part10_benchmark_cap_sensitivity/colab_part10_seed42',
    'outputs/part10_benchmark_cap_sensitivity_outputs/part10_benchmark_cap_sensitivity/colab_part10_seed42',
    'outputs/part10_benchmark_cap_sensitivity_outputs/part10_benchmark_cap_sensitivity_outputs/part10_benchmark_cap_sensitivity/colab_part10_seed42',
)
PART5_RUN_DIR = choose_run_dir(
    'colab_part5_seed42',
    ['run_manifest.json', 'results/output_validation_summary.json', 'results/rebalanced_performance_summary.csv'],
    'outputs/part5_implementability_rebalancing/colab_part5_seed42',
    'outputs/part5_implementability_rebalancing/part5_implementability_rebalancing/colab_part5_seed42',
    'outputs/part5_implementability_rebalancing_outputs/part5_implementability_rebalancing/colab_part5_seed42',
    'outputs/part5_implementability_rebalancing_outputs/part5_implementability_rebalancing_outputs/part5_implementability_rebalancing/colab_part5_seed42',
)
OUTPUT_DIR = Path('outputs/part13_transaction_cost_sensitivity_outputs/part13_transaction_cost_sensitivity')
RUN_ID = 'colab_part13_seed42'

print('INPUT_DIR:', INPUT_DIR)
print('PART10_RUN_DIR:', PART10_RUN_DIR)
print('PART5_RUN_DIR:', PART5_RUN_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
assert PART10_RUN_DIR.exists(), 'Part 10 run directory not found. Run Part 10 first.'
assert PART5_RUN_DIR.exists(), 'Part 5 run directory not found. Run Part 5 first.'

In [ ]:
!python experiments/part13_transaction_cost_sensitivity/run_part13_transaction_cost_sensitivity.py \
  --input-dir "{INPUT_DIR}" \
  --part10-run-dir "{PART10_RUN_DIR}" \
  --part5-run-dir "{PART5_RUN_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --run-id "{RUN_ID}" \
  --seed 42

In [ ]:
import json
import pandas as pd

RUN_DIR = OUTPUT_DIR / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
summary = pd.read_csv(RESULTS / 'part13_implementability_cost_summary.csv')
display(pd.read_csv(RESULTS / 'part13_cost_scenario_dictionary.csv'))
display(summary[
    (summary['funding_convention'] == 'bil_sleeve')
    & (summary['rebalance_frequency'] == 'monthly')
    & (summary['cost_scenario'] == 'medium_cost')
    & (summary['include_initial_formation_cost'] == True)
][[
    'rule_id', 'portfolio_family', 'total_transaction_cost', 'initial_formation_cost',
    'final_cumulative_net_value', 'net_annualized_mean_arithmetic', 'net_annualized_volatility'
]])
display(pd.read_csv(RESULTS / 'part13_cost_impact_comparison.csv').head(30))

In [ ]:
findings = json.loads((RESULTS / 'part13_key_findings.json').read_text())
print(json.dumps(findings, indent=2))

In [ ]:
from IPython.display import Image, display

for name in [
    'part13_cost_sensitivity_net_mean.png',
    'part13_cost_sensitivity_total_cost.png',
    'part13_cost_sensitivity_turnover.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
import shutil

zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Created zip:', zip_path)